In [1]:
import os
import sys
import pandas as pd

sys.path.append(os.path.join(os.getcwd(), ".."))

from src.feature_engineering import tfidf_vectorize, save_vectorizer, get_top_features

BASE_DIR = os.path.dirname(os.getcwd())
PROC_DIR = os.path.join(BASE_DIR, "data", "processed")
MODELS_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

In [2]:
# News Headlines
hl_train = pd.read_csv(os.path.join(PROC_DIR, "headlines_train.csv"))
hl_test  = pd.read_csv(os.path.join(PROC_DIR, "headlines_test.csv"))

# Reddit SARC
rd_train = pd.read_csv(os.path.join(PROC_DIR, "reddit_train.csv"))
rd_test  = pd.read_csv(os.path.join(PROC_DIR, "reddit_test.csv"))

# SemEval
se_train = pd.read_csv(os.path.join(PROC_DIR, "semeval_train.csv"))
se_test  = pd.read_csv(os.path.join(PROC_DIR, "semeval_test.csv"))

print("Headlines  — train:", len(hl_train), " test:", len(hl_test))
print("Reddit     — train:", len(rd_train), " test:", len(rd_test))
print("SemEval    — train:", len(se_train), " test:", len(se_test))

Headlines  — train: 22895  test: 5724
Reddit     — train: 808612  test: 202153
SemEval    — train: 3052  test: 764


In [3]:
# NEWS HEADLINES — TF-IDF

hl_X_train, hl_X_test, hl_vectorizer = tfidf_vectorize(
    hl_train["text"],
    hl_test["text"],
    ngram_range=(1, 2),
    max_features=5000
)

print("Headlines TF-IDF matrix shape:")
print("  Train:", hl_X_train.shape)
print("  Test: ", hl_X_test.shape)

print("\nTop 20 features:")
print(get_top_features(hl_vectorizer, n=20))

save_vectorizer(hl_vectorizer, os.path.join(MODELS_DIR, "tfidf_headlines.pkl"))

Headlines TF-IDF matrix shape:
  Train: (22895, 5000)
  Test:  (5724, 5000)

Top 20 features:
['000' '000 in' '10' '10 000' '10 things' '10 years' '100' '11' '12'
 '12 year' '13' '14' '15' '150' '16' '17' '18' '19' '20' '20 minutes']
Vectoriser saved: c:\Users\Nandita\Documents\Sarcasm_detection_in_social_media\models\tfidf_headlines.pkl


In [4]:
# REDDIT SARC — TF-IDF

# Text only
rd_X_train, rd_X_test, rd_vectorizer = tfidf_vectorize(
    rd_train["text"],
    rd_test["text"],
    ngram_range=(1, 2),
    max_features=5000
)

print("Reddit TF-IDF (text only):")
print("  Train:", rd_X_train.shape)
print("  Test: ", rd_X_test.shape)

print("\nTop 20 features:")
print(get_top_features(rd_vectorizer, n=20))

save_vectorizer(rd_vectorizer, os.path.join(MODELS_DIR, "tfidf_reddit.pkl"))

Reddit TF-IDF (text only):
  Train: (808612, 5000)
  Test:  (202153, 5000)

Top 20 features:
['000' '10' '10 10' '10 years' '100' '1000' '11' '12' '13' '14' '15' '16'
 '17' '18' '1st' '20' '200' '2014' '2015' '2016']
Vectoriser saved: c:\Users\Nandita\Documents\Sarcasm_detection_in_social_media\models\tfidf_reddit.pkl


In [5]:
# REDDIT SARC — TF-IDF
#   Text with context (parent comment + [SEP] + reply) --
rd_X_train_ctx, rd_X_test_ctx, rd_vectorizer_ctx = tfidf_vectorize(
    rd_train["text_with_context"],
    rd_test["text_with_context"],
    ngram_range=(1, 2),
    max_features=5000
)

print("Reddit TF-IDF (with context):")
print("  Train:", rd_X_train_ctx.shape)
print("  Test: ", rd_X_test_ctx.shape)

save_vectorizer(rd_vectorizer_ctx, os.path.join(MODELS_DIR, "tfidf_reddit_context.pkl"))

Reddit TF-IDF (with context):
  Train: (808612, 5000)
  Test:  (202153, 5000)
Vectoriser saved: c:\Users\Nandita\Documents\Sarcasm_detection_in_social_media\models\tfidf_reddit_context.pkl


In [6]:
# SEMEVAL — TF-IDF

se_X_train, se_X_test, se_vectorizer = tfidf_vectorize(
    se_train["text"],
    se_test["text"],
    ngram_range=(1, 2),
    max_features=5000
)

print("SemEval TF-IDF matrix shape:")
print("  Train:", se_X_train.shape)
print("  Test: ", se_X_test.shape)

print("\nTop 20 features:")
print(get_top_features(se_vectorizer, n=20))

save_vectorizer(se_vectorizer, os.path.join(MODELS_DIR, "tfidf_semeval.pkl"))

SemEval TF-IDF matrix shape:
  Train: (3052, 5000)
  Test:  (764, 5000)

Top 20 features:
['00' '000' '10' '100' '102' '11' '12' '13' '14' '15' '150' '16' '17' '19'
 '1am' '20' '200' '2014' '2015' '21']
Vectoriser saved: c:\Users\Nandita\Documents\Sarcasm_detection_in_social_media\models\tfidf_semeval.pkl


In [7]:
print("FEATURE ENGINEERING COMPLETE")

datasets = [
    ("Headlines",        hl_X_train, hl_X_test),
    ("Reddit (no ctx)",  rd_X_train, rd_X_test),
    ("Reddit (ctx)",     rd_X_train_ctx, rd_X_test_ctx),
    ("SemEval",          se_X_train, se_X_test),
]

for name, X_tr, X_te in datasets:
    print(f"\n{name}")
    print(f"  Train matrix : {X_tr.shape}")
    print(f"  Test  matrix : {X_te.shape}")
    print(f"  Vocabulary   : {X_tr.shape[1]} features")

print("\nVectorisers saved to:", MODELS_DIR)

FEATURE ENGINEERING COMPLETE

Headlines
  Train matrix : (22895, 5000)
  Test  matrix : (5724, 5000)
  Vocabulary   : 5000 features

Reddit (no ctx)
  Train matrix : (808612, 5000)
  Test  matrix : (202153, 5000)
  Vocabulary   : 5000 features

Reddit (ctx)
  Train matrix : (808612, 5000)
  Test  matrix : (202153, 5000)
  Vocabulary   : 5000 features

SemEval
  Train matrix : (3052, 5000)
  Test  matrix : (764, 5000)
  Vocabulary   : 5000 features

Vectorisers saved to: c:\Users\Nandita\Documents\Sarcasm_detection_in_social_media\models
